In [315]:
import gpt as g
import sys, os
import numpy as np

from scipy.linalg import expm
import time
import matplotlib.pyplot as plt
from gpt.qcd.gauge.smear import local_stout  

import time

In [316]:
local_stout_parallel_projector = g.default.get_int("--local-stout-parallel-projector", 1)

plaquette_stencil_cache = {}


def create_adjoint_projector(D, B, generators, nfactors):
    ng = len(generators)
    code = []
    idst = 0
    if nfactors == 1:
        iB = 1
    else:
        iA = 1
        iB = 2
    igen = iB + 1
    ndim = B.otype.shape[0]
    ti = g.stencil.tensor_instructions
    for c in range(ng):
        if local_stout_parallel_projector:
            itmp1 = -(igen + ng + 2 * c)
            itmp2 = -(igen + ng + 2 * c + 1)
        else:
            itmp1 = -(igen + ng)
            itmp2 = -(igen + ng + 1)
        # itmp1 = 2j * generators[c] * B
        imm = itmp1 if nfactors == 1 else itmp2
        ti.matrix_multiply(code, ndim, 2j, imm, igen + c, iB)
        if nfactors == 2:
            ti.matrix_multiply(code, ndim, 1.0, itmp1, iA, itmp2)
        # itmp2 = 0.5 * itmp1 - 0.5 * adj(itmp1)
        ti.matrix_anti_hermitian(code, ndim, itmp2, itmp1)
        # itmp1[0,0] = g.trace(itmp2) / 3.0
        ti.matrix_trace(code, ndim, 0, 1.0 / 3.0, itmp1, itmp2)
        # itmp2[i,i] -= itmp1[0,0]
        ti.matrix_diagonal_subtract(code, ndim, itmp2, itmp1)
        # now Dprime[d, c] = g(-g.trace(1j * generators[d] * itmp2))
        for d in range(ng):
            dst = d * ng + c
            # ti.matrix_trace_ab(code, ndim, dst, -1j, idst, igen + d, itmp2)
            # ti.matrix_trace_ab(code, ndim, dst, -1j, idst, itmp2, igen + d)
            ti.matrix_trace_ab_sparseb(
                code, ndim, dst, -1j, idst, itmp2, -(igen + d), generators[d]
            )

    if local_stout_parallel_projector:
        segments = [(len(code) // ng, ng)]
    else:
        segments = [(len(code) // 1, 1)]
    ein = g.stencil.tensor(D, [(0, 0, 0, 0)], code, segments)

    nx = 2 * ng if local_stout_parallel_projector else 2
    fgenerators = [g.lattice(B) for d in range(ng + nx)]
    for d in range(ng):
        fgenerators[d][:] = generators[d]

    return ein, fgenerators


def adjoint_from_right_fast(D, UtaU, generators, cache):
    if "stencil" not in cache:
        cache["stencil"] = create_adjoint_projector(D, UtaU, generators, 1)

    ein, fgenerators = cache["stencil"]

    ein(D, UtaU, *fgenerators)


def compute_adj_ab(A, B, C, generators, cache):
    if "stencil_ab" not in cache:
        cache["stencil_ab"] = create_adjoint_projector(C, A, generators, 2)

    ein, fgenerators = cache["stencil_ab"]

    ein(C, g(g.adj(A)), g(B), *fgenerators)


def compute_adj_abc(_A, _B, _C, _V, generators, cache, parity):

    t = g.timer("compute_adj_abc")
    t("checkerboarding")
    A = g.pick_checkerboard(parity, _A)
    B = g.pick_checkerboard(parity, _B)
    C = g.pick_checkerboard(parity, _C)
    V = g.pick_checkerboard(parity, _V)

    t("other")
    ng = len(generators)
    tmp2 = {}
    D = g.lattice(C)
    for a in range(ng):
        UtaU = g(g.adj(A) * 2j * generators[a] * B)

        # move the loop below to a stencil.tensor
        t("adj_from_right")
        adjoint_from_right_fast(D, UtaU, generators, cache)

        t("other")
        tmp2[a,] = g(g.trace(C * D))
    t("merge")
    g.merge_color(V, tmp2)
    t("checkerboarding")

    _V[:] = 0
    g.set_checkerboard(_V, V)

    t()
    # g.message(t)


def csf(link, mu, field=None):
    if field is None:
        field = g.identity(link)
    return link * g.cshift(field, mu, 1)


def csb(link, mu, field=None):
    if field is None:
        field = g.identity(link)
    return g.cshift(g.adj(link) * field, mu, -1)


def adjoint_to_fundamental(fund, adj, generators):
    ng = len(generators)
    fund[:] = 0
    adj_c = g.separate_color(adj)
    for e in range(ng):
        fund += 1j * adj_c[e,] * generators[e]

Jacobian calculation
=

\begin{equation}
\begin{aligned}
    \Big[\mathcal{F}(x)\Big](z,\rho; y,\nu)^{ab} & = (\text{unupdated links}) \delta_{yz} \delta_{\nu\rho} \delta^{ab} \\
    & +  \delta_{xy} J(-X)^{ab} \Big\{ \Big[J(X)^{-1}\Big] ^{bc} \delta_{yz}\delta_{\nu\rho} - t \Big[O_*(U)\Big](y, \nu; z,\rho)^{bc} \Big\}_{X = \tau \Omega(x)} \\
    & +\delta_{y,x-\hat{\nu}}\Bigg[ \delta^{ab}\delta_{zy}\delta_{\rho \nu} - tJ(X)^{bf} \Big[O_*(U)\Big](y, \nu; z,\rho)^{fd} \Big(\adj_{U}\Big)^{da} \Bigg]_{X = \tau \Omega}. \\
\end{aligned}
\end{equation}

\begin{equation}
    \partial^c_{z,\rho} \Omega(y) = \mathcal{P} \Big( T^c U(y,\nu) \Big)\delta_{x,y}\delta_{\rho,\nu} + \mathcal{P} \Big( U^\dag(y-\hat{\nu},\nu) T^c \Big)\delta_{z,y-\hat{\nu}}\delta_{\rho,\nu}.
\end{equation}

- Make_diff_exp_map (done)

- make derivatives of $X^{A}_B$ function (done)

tricky bit now it to actually construct this matrix from all of the components!

In [317]:
def div_A(U):
    a = g.qcd.gauge.fix.landau(U)
    V = g.mcolor(grid) # V is a SU(3) matrix on every lattice site
    rng.normal_element(V, scale=0.00)
    return a.gradient(V, 0.) * 1j

def div_A2(U):
    # code below comes from lib/gpt/qcd/gauge/fix/landau.py
    V = g.mcolor(grid)
    rng.normal_element(V, scale=0.00)
    
    A = [
            g.qcd.gauge.project.traceless_anti_hermitian(u) / 1j
            for u in U
        ]
    
    dmuAmu = V.new()
    dmuAmu.otype = V.otype.cartesian()
    dmuAmu = g(0.0 * dmuAmu)
    #for mu, Amu in enumerate(A):
    for mu in [0,1,2,3,]:
        # I don't actually want to append here, I really want the sum over mu
        dmuAmu += A[mu] - g.cshift(A[mu], mu, -1)
    return dmuAmu

def exp_div_A(U, delta_t=0.05):
    # replace with Cayley-Hamilton
    a = g.matrix.exp(  - delta_t * div_A2(U))
    #V = g.mcolor(grid)
    #a.otype = V.otype
    return a

def exp_div_A2(U, delta_t=0.05):
    #V = g.mcolor(grid)
    #f = [u for u in div_A2(U)]
    #for mu in[0,1,2,3]:
    #    f.otype = V.otype
    
    #a = [ g.matrix.exp(  - delta_t * u) for u in div_A2(U)]
    return g.matrix.exp(  - delta_t * div_A2(U))
    
def exp_div_A_cayley_Hamilton(U, delta_t=0.05):
    # slower, presumably because of grad
    divA = div_A2(U)
    a = g.matrix.exp.function_and_gradient(  - delta_t * divA, divA)
    #V = g.mcolor(grid)
    #a.otype = V.otype
    return a


def gtf(U, rho=0.05):
    V = exp_div_A(U, rho)
    return g.qcd.gauge.transformed(U,V)

def trace_U(U):                                                      
    return sum(v for v in sum(u[:].real for u in g.eval(g.trace(U)))) / (4 * size**4 ) / 3.

In [318]:
num_steps = 1
def ftg(U, eps):
    global num_steps

    ##### dmuAmu ##############
    #B = U[0] -  g.adj(g.cshift(U[0], 0, -1))
    B = U[0] -  g.cshift(U[0], 0, -1)
    for mu in [1,2,3,]:
        #B += U[mu] -  g.adj(g.cshift(U[mu], mu, -1))
        B += U[mu] -  g.cshift(U[mu], mu, -1)
        

   #### masks for all even/odd sites ########

    grid_cb = grid.checkerboarded(g.redblack)
    one_cb = g.complex(grid_cb)
    one_cb[:] = 1

    masks = {}
    for p in [g.even, g.odd]:
        m = g.complex(grid)
        m[:] = 0
        one_cb.checkerboard(p)
        g.set_checkerboard(m, one_cb)
        masks[p] = m
    
    if num_steps // 2 == 0:
        mask, imask = masks[g.odd], masks[g.odd.inv()] 
    else:
        mask, imask = masks[g.even], masks[g.even.inv()]
    
    num_steps += 1
    fm = g(mask + 1e-15 * imask)

    ###### apply masks ########
    B *= fm

   # apply gtf 
    U_prime = []
    for mu in [0,1,2,3]:
        U_mu_prime = g(
                g.matrix.exp(  - eps * g.qcd.gauge.project.traceless_anti_hermitian(B) ) 
                * U[mu] * g.matrix.exp(  + eps * g.qcd.gauge.project.traceless_anti_hermitian( g.cshift(B, mu, +1) ) ) 
        )
        U_prime.append(U_mu_prime)
    
    return U_prime


In [319]:
size = 4
grid = g.grid([size, size, size, size], g.double)
rng = g.random("t")

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)

trace_U(U)

GPT :   75228.940205 s : Initializing gpt.random(t,vectorized_ranlux24_389_64) took 0.000402927 s


array([0.45075534])

In [321]:
g.eval(div_A(U))

lattice(ot_matrix_su_n_fundamental_algebra(3),double)

In [ ]:
# working inv
def inv(fields, max_iter=15):
    U_prime = g.copy(fields)
    for it in range(max_iter):
        U_prime_mu_last = g.copy(U_prime)
        gtf, UU, fm = get_gtf(U_prime)
        U_prime = g.qcd.gauge.transformed(U_prime_2, g.matrix.exp(  gtf))
    return U_prime

In [ ]:
dft = g.qcd.gauge.smear.differentiable_field_transformation(
    U,
    ft_stout,
    # g.algorithms.inverter.fgmres(eps=1e-15, maxiter=1000, restartlen=60),
    # g.algorithms.inverter.fgmres(eps=1e-15, maxiter=1000, restartlen=60),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=1000, restartlen=60),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=1000, restartlen=60),
    g.algorithms.optimize.non_linear_cg(
        maxiter=1000, eps=1e-15, step=1e-1, line_search=ls2, beta=fr
    ),
)

In [ ]:
# working gtf
def get_gtf(fields, rho=.05):
    grid = fields[0].grid
    #rho = 0.05
    nd = grid.nd
    U = fields[0:nd]
    #if grid in self.cache:
    #    masks = self.cache[grid]
    #else:
    grid_cb = grid.checkerboarded(g.redblack)
    one_cb = g.complex(grid_cb)
    one_cb[:] = 1

    masks = {}
    for p in [g.even, g.odd]:
        m = g.complex(grid)
        m[:] = 0
        one_cb.checkerboard(p)
        g.set_checkerboard(m, one_cb)
        masks[p] = m

        #self.cache[grid] = masks

    #///////////////////////////////////////////////////////////
    # be careful, is the below actually putting the gauge 
    # transformation on odd and even lattice sites like you want????
    #///////////////////////////////////////////////////////////
    
    #mask, imask = masks[self.params["checkerboard"]], masks[self.params["checkerboard"].inv()]
    mask, imask = masks[g.even], masks[g.even.inv()] 
    fm = g(mask + 1e-15 * imask)
    gt = rho * div_A(fields)
    
    return g(gt * fm), U, fm

In [ ]:
size = 4
grid = g.grid([size, size, size, size], g.double)
rng = g.random("t")

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)

fields = U

grid = fields[0].grid
nd = grid.nd

U[0]

In [ ]:
gtf, U, fm = get_gtf(U)
U_prime_2 = g.qcd.gauge.transformed(U, g.matrix.exp(  - gtf))
U_prime_3 = g.qcd.gauge.transformed(U, g.matrix.exp.function_and_gradient(  - gtf, div_A(U))[0])
U_prime_2[:][0][0,0,0,0] - U_prime_3[:][0][0,0,0,0]

In [ ]:
# working inv
def inv(fields, max_iter=15):
    U_prime = g.copy(fields)
    for it in range(max_iter):
        U_prime_mu_last = g.copy(U_prime)
        gtf, UU, fm = get_gtf(U_prime)
        U_prime = g.qcd.gauge.transformed(U_prime_2, g.matrix.exp(  gtf))
    return U_prime

In [10]:
# computing J(X) complete

#In GPT for stout there is a function for full jacobian and jacobian component function for just diag bit needed for log det
# derivative of exp map gonna be giga helpful??? /lib/gpt/core/foundation/lattice/matrix/exp.py > cayley_hamilton_function_and_gradient_3

#jacobian_components(self, fields, cache_ab):
gtf, U, fm =  get_gtf(U)

#U_mu = U[mu]

grid = U[0].grid
dt = grid.precision.complex_dtype
otype = fields[0].otype
cartesian_otype = otype.cartesian()
adjoint_otype = g.ot_matrix_su_n_adjoint_algebra(otype.Nc)
generators = cartesian_otype.generators(dt)
ng = len(generators)

N_cb = g.lattice(grid, adjoint_otype) # X_* in my language, but I'll need to compute this for several functions locations simultaneously
Z_ac = g.lattice(grid, adjoint_otype) # Omega in my language

adjoint_generators = adjoint_otype.generators(dt)

#M = g(U_mu * g.adj(C_mu))
#M = g()

adj_id = g.identity(g.lattice(grid, adjoint_otype))
fund_id = g.identity(g.lattice(grid, otype))

#compute_adj_ab(fund_id, M, N_cb, generators, cache_ab)

#Z = g(g.qcd.gauge.project.traceless_anti_hermitian(g.adj(M)))

# below is P{U (staple)} in adjoint rep
####### orig #########
#Z_ac[:] = 0
#for b in range(ng):
#    coeff = g(2 * g.trace(1j * generators[b] * Z)) # tr T^a [(M - M^\dag) - (1/3)tr(M - M^\dag)] = tr T^a [M - M^\dag] = Retr(T^a M)
#    Z_ac += 1j * adjoint_generators[b] * coeff
######################


# gtf version
Z = g(gtf)
Z_ac[:] = 0
for b in range(ng):
    coeff = g(2 * g.trace(1j * generators[b] * Z)) # Retr(T^a M) consider P{M} = \sum_a Retr(T^a M) T^a
    Z_ac += 1j * adjoint_generators[b] * coeff # to adj rep

# compute J
# Need both J(- X_L) and J_(X_R) 
# consider X_L = -dmuAmu(x), X_R = dmuAmu
# => Just compute J same as prev.

X = g.copy(adj_id)
J_ac = g.copy(adj_id)
kpfac = 1.0
denom = g.norm2(X)
nmax = 25
for k in range(1, nmax):
    X @= X * Z_ac
    kpfac = kpfac / (k + 1)
    Y = g(X * kpfac)
    eps = (g.norm2(Y) / denom) ** 0.5
    J_ac += Y
    if eps < grid.precision.eps:
            break
assert k != nmax - 1

    # combined M
#EM_ab = g(adj_id - J_ac * N_cb)

    # return component
#return J_ac, N_cb, Z_ac, M, fm, M_ab
J_ac[:] # correctly isn't zero everywhere because every link is hit with just even site update -e-o-e-

#np.linalg.inv(J_ac[:][0]) @ J_ac[:][0] # need to use @ for these

array([[[ 9.85665114e-01+0.j,  1.33076963e-01+0.j, -5.04913728e-02+0.j,
         ..., -1.31041426e-02+0.j,  8.60617628e-03+0.j,
          1.02118192e-03+0.j],
        [-1.36995772e-01+0.j,  9.84139669e-01+0.j, -6.10924227e-02+0.j,
         ...,  1.15001872e-02+0.j,  1.25964105e-02+0.j,
          1.93982149e-04+0.j],
        [ 3.86135974e-02+0.j,  6.92157928e-02+0.j,  9.95110023e-01+0.j,
         ..., -1.77825066e-02+0.j,  2.17820589e-02+0.j,
          5.68426390e-04+0.j],
        ...,
        [ 1.68797812e-02+0.j, -2.30674156e-03+0.j,  2.13187390e-02+0.j,
         ...,  9.80375427e-01+0.j, -1.58989820e-01+0.j,
         -3.78883615e-02+0.j],
        [-5.09952359e-03+0.j, -1.72259174e-02+0.j, -1.48754745e-02+0.j,
         ...,  1.60950671e-01+0.j,  9.80521393e-01+0.j,
          2.88243209e-02+0.j],
        [ 9.89992760e-04+0.j,  4.51835756e-04+0.j,  4.68304318e-04+0.j,
         ...,  3.07574680e-02+0.j, -3.47121125e-02+0.j,
          9.97947060e-01+0.j]],

       [[ 1.00000000e+00+0.j,  

\begin{equation}
    \partial^c_{z,\rho} \Omega(y) = \mathcal{P} \Big( T^c U(y,\nu) \Big)\delta_{x,y}\delta_{\rho,\nu} + \mathcal{P} \Big( U^\dag(y-\hat{\nu},\nu) T^c \Big)\delta_{z,y-\hat{\nu}}\delta_{\rho,\nu}.
\end{equation}

In [29]:
# compute X^R_{R*}

# fund_id = g.identity(g.lattice(grid, otype)) # already defined above
# generators = cartesian_otype.generators(dt) # " "

start = time.time()

mk_XAB = np.frompyfunc(lambda _: g.copy(U), 1, 1)
dummy = np.zeros((8, 4))

XRR_prime = mk_XAB(dummy)
XLL_prime = mk_XAB(dummy)
XLL_prime_int = mk_XAB(dummy)

for b in range(ng):
    for mu in range(4):
        XRR_prime[b][mu] = g(g.qcd.gauge.project.traceless_anti_hermitian(1j*generators[b] * U[mu]))

for b in range(ng):
    for mu in range(4):
        XLL_prime_int[b][mu] = g(g.qcd.gauge.project.traceless_anti_hermitian( U[mu]* 1j *generators[b]))
        
for b in range(ng):
    for mu in range(4):
        XLL_prime[b][mu] = g.cshift(XLL_prime_int[b][mu], mu, -1)

XLR_prime = - XLL_prime

# might not need these bits...

dummy = np.zeros((8, 8, 4))
XRR_prime_cd = mk_XAB(dummy)
XLL_prime_cd = mk_XAB(dummy)


for c in range(ng):
    for d in range(ng):
        for mu in range(4):
            XRR_prime_cd[c][d][mu] = g.trace(2 * generators[d] * XRR_prime[c][mu])

for c in range(ng):
    for d in range(ng):
        for mu in range(4):
            XLL_prime_cd[c][d][mu] = g.trace(2 * generators[d] * XLL_prime[c][mu])

XRL_prime_cd = - XRR_prime_cd
XLR_prime_cd = - XLL_prime_cd

#XRR_cd = np.array(XRR_cd)
# XLL_prime = g.cshift(XRR_prime[:][mu], mu, -1)
# XLR_pime = - XLL_prime 


end = time.time()

#print(end - start)

In [30]:
XRR_ac = g.lattice(grid, adjoint_otype) # Omega in my language
adjoint_generators = adjoint_otype.generators(dt)

for b in range(ng):
    coeff = g(2* g.trace(1j * generators[b] * \
                         g.qcd.gauge.project.traceless_anti_hermitian(1j*generators[b] * U)))
    
    
    #    Z_ac += 1j * adjoint_generators[b] * coeff # to adj rep       
        
#XRL_prime = - XRR_prime


TypeError: 'tensor' object cannot be interpreted as an integer

In [31]:
# assemble Jacobian
#a = g.ot_matrix_real_additive_group(8)
#c =  g.lattice(grid, a)
#c[:] *= 0

# assign the links to these 64x64 blocks in a nice way....

#c[:][1,:,:].shape
#z = np.eye((64))
#c[:] += z
#detc = g.matrix.det(c)
#detc[:]
#c[:].shape

#detJ = np.zeros((64,64))

#for i in range(ng):
XRR_prime[:].shape

(8, 4)

\begin{equation}
\begin{aligned}
    \Big[\mathcal{F}(x)\Big](y,\nu; z,\rho)^{ad}  =  J^{ac}(X_R) \delta_{(y,\nu), RI}  \Bigg\{  & \delta_{(z,\rho), LI} X^L_{L*}(z,\rho; y,\nu)^{cd} \\ 
    & +\delta_{(z,\rho), RI} \Big[ X^R_{L*}(z,\rho; y,\nu)^{cd} + \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{z,y} \delta_{\nu, \rho}\Big] \Bigg\} \\
    + J^{ac}(X_R) \delta_{(y,\nu), LI} \Bigg\{ & \delta_{(z,\rho), LI}  \Big[ X_{R*}^L (z,\rho; y, \nu)^{cd} - \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{y,z} \delta_{\rho,\nu}\Big] \\
    & +\delta_{(z,\rho), RI} X_{R*}^R (z,\rho; y, \nu)^{cd} \Bigg\}.
\end{aligned}
\end{equation}

Is it even worth constructing the full matrix, or would it be better to just keep it in components and use them appropriately? Determinant might be a bit difficult to calculate.

Use
$$
RI = (x,\mu)
$$
$$
LI = (x-\hat{\mu},\mu)
$$
Then we can construct

\begin{equation}
\begin{aligned}
    \Big[\mathcal{F}(x)\Big](y,\nu; z,\rho)^{ad}  =  J^{ac}(X_R)  \Bigg\{  &  X^L_{L*}(x-\hat{\mu},\mu; x,\nu)^{cd} \\ 
    & + \Big[ X^R_{L*}(x,\mu; x,\nu)^{cd} + \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{x,x} \delta_{\mu, \nu}\Big] \Bigg\} \\
    + J^{ac}(X_R) \Bigg\{ &  \Big[ X_{R*}^L (x - \hat{\mu},\mu; x-\hat{\nu}, \nu)^{cd} - \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{x-\hat{\mu},x-\hat{\nu}} \delta_{\mu,\nu}\Big] \\
    & + X_{R*}^R (x,\mu; x-\hat{\nu}, \nu)^{cd} \Bigg\}.
\end{aligned}
\end{equation}

rearrange

\begin{equation}
\begin{aligned}
    \Big[\mathcal{F}(x)\Big](y,\nu; z,\rho)^{ad}  =  J^{ac}(X_R)  \Bigg\{  & - \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{x-\hat{\mu},x-\hat{\nu}} \delta_{\mu,\nu} +  X_{R*}^L (x - \hat{\mu},\mu; x-\hat{\nu}, \nu)^{cd} + X^L_{L*}(x-\hat{\mu},\mu; x,\nu)^{cd} \Bigg\}\\
    + J^{ac}(X_R) \Bigg\{ &   + \Big[J(-X_R)^{-1}\Big]^{cd} \delta_{x,x} \delta_{\mu, \nu}+ X^R_{L*}(x,\mu; x,\nu)^{cd}  + X_{R*}^R (x,\mu; x-\hat{\nu}, \nu)^{cd} \Bigg\}. \\
\end{aligned}
\end{equation}

Algorithm for applying this Jacobian without flatten and apply?

\begin{pmatrix}
\cdot & - & - & - \\
| & (x,\mu)^{ad} &  &  \\
| &  & (x-\hat{\mu}, \mu)^{ad} &  \\
| &  &  & \cdot \\
\end{pmatrix}

Make matrix that just gives indices for these cshifts

$$
F(x,\mu)^{ac} \rightarrow \sum_{z,\nu} [J(x)]^{ad}(x,\mu; z,\nu) F(z,\nu)^{dc} + (\text{grad} \text{ log det}(\text{next Jacobian}))^{ac}(x,\mu)
$$ 

Therefore I want
- Force $\leftarrow$ (Forces at neighboring links) $\cdot$ (row in Jacobian) + grad log det $J_{\text{new}}$

Issue is that when I want to evaluate the forces of the new log det Jacobian I need

$$
d \log \det F = \text{tr}(F^{-1} dF)
$$

So going to have to flatten the Jacobian anyway...

In [34]:
# is this algorithm just way too complicated to implement? Whatever, even if its complicated as shit and too slow just 
# implement it, publish the paper and be done with it if needed.

# chat gpt says the number of flops required to invert the 64x64 matrix is actually not very large so this shouldnt be an issue

In [35]:
B = np.zeros((64,8,8))



blocks = B.reshape(8, 8, 8, 8)  # (grid_rows, grid_cols, block_rows, block_cols)
rearranged = blocks.transpose(0, 2, 1, 3)  # now each block row is adjacent
result = rearranged.reshape(64, 64)

# Let's check the shape and content
print("Shape of the resulting array:", result.shape)
print("Resulting array:\n", result)

Shape of the resulting array: (64, 64)
Resulting array:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


$$
d \log \det F = \text{tr}(F^{-1} dF)
$$
$$
F = F(dJ, d\Omega, e^{\text{adj}\Omega })
$$

Easy because I can compute this without much difficulty
$$
d\Omega
$$

Easy because I can just compute the inverse matrix (it is rather large though...)
$$
F^{-1}
$$

Hard because J is a matrix?
$$
dJ = \frac{dJ}{dX}\frac{dX}{d\Omega^a}
$$

derivatives of adjoint action
 
\begin{equation}
\begin{aligned}
    d \Big( e^{A} T^a e^{-A} \Big) & = de^{A} T^c e^{-A} + e^{A} T^c de^{-A} \\
    & = de^A e^{-A} e^A T^c e^{-A} + e^A T^c e^{-A} e^A d e^{-A} \\
    & = J(A) A_* e^A T^c e^{-A} - e^A T^c e^{-A} J(-A) A_* \\
    & = \big\{J(A) \cdot A_*\big\}^l T^l \Big(e^{\adj{A}} \Big)^{ac} T^c - \Big(e^{\adj{A}} \Big)^{ac} T^c \big\{J(-A) \cdot A_*\big\}^l T^l \\
    & = \Big(e^{\adj{A}} \Big)^{ac} \Big[ \big\{J(A) \cdot A_*\big\}^l T^l T^c - T^c \big\{J(-A) \cdot A_*\big\}^l T^l \Big]
\end{aligned}
\end{equation}


Using prebuilt Cayley-Hamilton formula from GPT

$$
\frac{d}{dt} e^{A} = \frac{\partial e^A}{\partial A} \frac{\partial A}{\partial U} \frac{dU}{dt} = \Big(J(A) e^A \Big)(Q_*)\Big(\frac{dU}{dt} \Big)
$$

$$
\Big( \frac{d}{dt} e^{A} \Big)_{ij} = (J(A) e^A)_{ij}
$$